# EnergyCraft FWI — Google Colab

Запускайте ячейки **сверху вниз**. Сессия Colab временная — при каждом новом запуске повторите клонирование и загрузку данных.

**Runtime → Change runtime type → GPU** (рекомендуется для CNN/PINN).

## 0. Конфигурация

In [ ]:
# >>> ЗАМЕНИТЕ на свой репозиторий <<<
GITHUB_REPO = "https://github.com/<USER>/EnergyCraft.git"
BRANCH = "main"

# Папка на Google Drive с данными (SEG-Y или уже data/processed)
DRIVE_DATA_DIR = "/content/drive/MyDrive/EnergyCraft/data"

# Куда клонировать проект в Colab
PROJECT_DIR = "/content/EnergyCraft"

## 1. Клонирование репозитория

In [ ]:
import os
import subprocess

def run(cmd, cwd=None):
    print(f"$ {cmd}")
    subprocess.check_call(cmd, shell=True, cwd=cwd)

if os.path.exists(PROJECT_DIR):
    run(f"git -C {PROJECT_DIR} pull")
else:
    run(f"git clone -b {BRANCH} {GITHUB_REPO} {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("Project dir:", os.getcwd())

## 2. Установка зависимостей

In [ ]:
run("pip install -q -r requirements.txt")

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Google Drive — загрузка данных

На Drive подготовьте структуру:

```
MyDrive/EnergyCraft/data/
  *.sgy                    # исходные SEG-Y
  processed/               # или уже конвертированные NumPy
    manifest.json
    *.npy
```

In [ ]:
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")

src = Path(DRIVE_DATA_DIR)
dst = Path(PROJECT_DIR) / "data"
dst.mkdir(parents=True, exist_ok=True)

if not src.exists():
    raise FileNotFoundError(
        f"Папка на Drive не найдена: {src}\n"
        "Создайте MyDrive/EnergyCraft/data и положите туда .sgy или processed/"
    )

# Копируем содержимое (не симлинк — надёжнее для больших файлов на Colab)
for item in src.iterdir():
    target = dst / item.name
    if item.is_dir():
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)

print("Data copied to:", dst)
print("Contents:", list(dst.iterdir())[:20])

## 4. Конвертация SEG-Y → NumPy (если ещё не сделано)

In [ ]:
from pathlib import Path

manifest = Path(PROJECT_DIR) / "data/processed/manifest.json"
if manifest.exists():
    print("manifest.json уже есть — пропускаем конвертацию")
else:
    run("python scripts/convert_sgy_to_numpy.py", cwd=PROJECT_DIR)

## 5. Запуск экспериментов

Раскомментируйте нужный блок. Параметры — в `src/config.py`.

In [ ]:
# --- CNN baseline (все датасеты) ---
run("python run_cnn_baseline.py", cwd=PROJECT_DIR)

# --- только Zoloto ---
# run("python run_cnn_baseline.py --dataset zoloto", cwd=PROJECT_DIR)

# --- PINN ---
# run("python run_pinn_fwi.py --max-windows 3", cwd=PROJECT_DIR)

# --- Diffusion (локальный) ---
# run("python diffusion_fwi/local_diffusion/train.py --max-epochs 30", cwd=PROJECT_DIR)

## 6. TensorBoard (онлайн в Colab)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/EnergyCraft/outputs/tensorboard --port 6006

## 7. Сохранение результатов обратно на Google Drive

In [ ]:
import shutil
from pathlib import Path
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_out = Path("/content/drive/MyDrive/EnergyCraft/runs") / stamp
drive_out.mkdir(parents=True, exist_ok=True)

for folder in ["models", "plots", "logs", "tensorboard"]:
    src = Path(PROJECT_DIR) / "outputs" / folder
    if src.exists():
        shutil.copytree(src, drive_out / folder)

for summary in Path(PROJECT_DIR).glob("outputs/summary_*.json"):
    shutil.copy2(summary, drive_out / summary.name)

print("Saved to:", drive_out)

## 8. Произвольная команда

Универсальная ячейка для любого скрипта проекта.

In [ ]:
SCRIPT = "python run_cnn_baseline.py --dataset domanic"
run(SCRIPT, cwd=PROJECT_DIR)